# Analyse Water Storage: Surface Waterbodies

List waterbodies and read their annual and seasonal water-spread areas from 2017–18 to 2024–25.

Run the cells in order. Each step uses data from the previous cells. You can edit the place, identifier, columns and chart settings as you go.


## Set up Python

Run these two collapsed cells once. They load the libraries and starting location. Expand them to see or change the setup.


In [ ]:
import sys
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["geopandas", "matplotlib", "requests", "pyodide-http"])
    import pyodide_http
    pyodide_http.patch_all()

import os
import ast
import json
from getpass import getpass
from urllib.parse import urljoin
import requests
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display, FileLink


In [ ]:
SCOPE = json.loads("{\"state\": \"Bihar\", \"district\": \"Nalanda\", \"tehsil\": \"Hilsa\"}")
GEOSERVER = 'https://geoserver.core-stack.org:8443/geoserver/'
API_URL = 'https://geoserver.core-stack.org/api/v1/'
STAC_URL = 'https://spatio-temporal-asset-catalog.s3.ap-south-1.amazonaws.com/CorestackCatalogs_merged_collection/tehsil_wise/catalog.json'
YEARS = list(range(2017, 2025))


## Choose the tehsil

The starting place is Hilsa, Nalanda, Bihar. A notebook downloaded from GeoLibre uses the selected tehsil. Change these names to read another place.


In [ ]:
state = SCOPE["state"].lower().replace(" ", "_")
district = SCOPE["district"].lower().replace(" ", "_")
tehsil = SCOPE["tehsil"].lower().replace(" ", "_")
place = {"state": state, "district": district, "tehsil": tehsil}
place


## The layers we will use

Each link reads a vector layer as GeoJSON. GeoJSON contains a feature list; each feature has a shape and its data fields.


In [ ]:
layer_urls = {
    "waterbodies": f"{GEOSERVER}swb/ows?service=WFS&version=2.0.0&request=GetFeature&typeNames=swb:surface_waterbodies_{district}_{tehsil}&outputFormat=application/json&srsName=EPSG:4326",
    "mws": f"{GEOSERVER}mws/ows?service=WFS&version=2.0.0&request=GetFeature&typeNames=mws:mws_{district}_{tehsil}&outputFormat=application/json&srsName=EPSG:4326",
    "rivers": f"{GEOSERVER}river/ows?service=WFS&version=2.0.0&request=GetFeature&typeNames=river:{district}_{tehsil}_river_vector&outputFormat=application/json&srsName=EPSG:4326",
    "canals": f"{GEOSERVER}canal/ows?service=WFS&version=2.0.0&request=GetFeature&typeNames=canal:{district}_{tehsil}_canal_vector&outputFormat=application/json&srsName=EPSG:4326",
    "stream_order": f"{GEOSERVER}stream_order/ows?service=WFS&version=2.0.0&request=GetFeature&typeNames=stream_order:stream_order_{district}_{tehsil}_vector&outputFormat=application/json&srsName=EPSG:4326",
}
pd.DataFrame(layer_urls.items(), columns=["Layer", "GeoJSON URL"])


## Read the surface-waterbody layer

`UID` identifies a waterbody. Annual `area_YY-YY` fields are hectares. `area_ored` is its combined detected footprint in hectares.


In [ ]:
waterbodies_response = requests.get(layer_urls["waterbodies"], timeout=90)
waterbodies_response.raise_for_status()
waterbodies_geojson = waterbodies_response.json()
waterbodies = gpd.GeoDataFrame.from_features(waterbodies_geojson["features"], crs="EPSG:4326")
waterbodies.drop(columns="geometry").head()


## Read the field descriptions

STAC records describe the published fields. This table selects the fields used below and keeps their original descriptions.


In [ ]:
item_name = f"{state}_{district}_{tehsil}_surface_water_bodies_vector"
item_url = urljoin(STAC_URL, f"{state}/{district}/{tehsil}/{item_name}/{item_name}.json")
item_response = requests.get(item_url, timeout=90)
item_response.raise_for_status()
item = item_response.json()
field_notes = pd.DataFrame(item["properties"]["table:columns"])
field_notes.loc[field_notes["name"].isin(['UID', 'area_ored', 'area_17-18', 'k_17-18', 'kr_17-18', 'krz_17-18']), ["name", "type", "description"]]


## Read MWS boundaries

These shapes provide the MWS context.


In [ ]:
mws_response = requests.get(layer_urls["mws"], timeout=90)
mws_response.raise_for_status()
mws_geojson = mws_response.json()
mws = gpd.GeoDataFrame.from_features(mws_geojson["features"], crs="EPSG:4326")
mws.drop(columns="geometry").head()


## Read rivers

Inspect the tehsil’s river features.


In [ ]:
rivers_response = requests.get(layer_urls["rivers"], timeout=90)
rivers_response.raise_for_status()
rivers_geojson = rivers_response.json()
rivers = gpd.GeoDataFrame.from_features(rivers_geojson["features"], crs="EPSG:4326")
rivers.drop(columns="geometry").head()


## Read canals

Inspect the tehsil’s canal features.


In [ ]:
canals_response = requests.get(layer_urls["canals"], timeout=90)
canals_response.raise_for_status()
canals_geojson = canals_response.json()
canals = gpd.GeoDataFrame.from_features(canals_geojson["features"], crs="EPSG:4326")
canals.drop(columns="geometry").head()


## Read stream-order shares

This vector layer describes stream-order area shares for each MWS.


In [ ]:
stream_order_response = requests.get(layer_urls["stream_order"], timeout=90)
stream_order_response.raise_for_status()
stream_order_geojson = stream_order_response.json()
stream_order = gpd.GeoDataFrame.from_features(stream_order_geojson["features"], crs="EPSG:4326")
stream_order.drop(columns="geometry").head()


## List the waterbody identifiers

Choose the first identifier, or copy another value from the list into `waterbody_id`.


In [ ]:
waterbody_ids = waterbodies["UID"].sort_values().tolist()
display(pd.DataFrame({"Waterbody identifier": waterbody_ids}))
waterbody_id = waterbody_ids[0]
waterbody = waterbodies.loc[waterbodies["UID"] == waterbody_id].iloc[0]
waterbody_id


## Read annual and seasonal values

The seasonal fields `k`, `kr` and `krz` are the Kharif, Rabi and Zaid percentages of `area_ored`. Keep these source percentages beside the hectare values.


In [ ]:
year_suffixes = [f"{y%100:02d}-{(y+1)%100:02d}" for y in YEARS]
water_area = pd.DataFrame({
    "Annual area (ha)": [waterbody[f"area_{year}"] for year in year_suffixes],
    "Kharif (%)": [waterbody[f"k_{year}"] for year in year_suffixes],
    "Rabi (%)": [waterbody[f"kr_{year}"] for year in year_suffixes],
    "Zaid (%)": [waterbody[f"krz_{year}"] for year in year_suffixes]
}, index=[f"{y}–{y+1}" for y in YEARS])
# Convert the recorded seasonal percentages to hectares using their reference footprint.
water_area["Kharif area (ha)"] = water_area["Kharif (%)"] * waterbody["area_ored"] / 100
water_area["Rabi area (ha)"] = water_area["Rabi (%)"] * waterbody["area_ored"] / 100
water_area["Zaid area (ha)"] = water_area["Zaid (%)"] * waterbody["area_ored"] / 100
water_area


## Total annual waterbody area

Sum each annual area column across the waterbody records. Each waterbody is counted once. These values describe water-spread area, not storage volume.


In [ ]:
annual_area_columns = [f"area_{year}" for year in year_suffixes]
total_area = waterbodies[annual_area_columns].sum(min_count=len(waterbodies))
total_area.index = water_area.index
display(total_area.to_frame("Total annual area (ha)"))
total_area.plot(marker="o", figsize=(10, 4), ylabel="Total waterbody area (ha)", xlabel="Year")
plt.show()


## Connect to the CoRE Stack API

The [API guide](https://api-doc.core-stack.org) explains access. The key goes in the `X-API-Key` header. This cell reads `CORE_STACK_API_KEY` from your environment, or asks for it without showing it. The key is not written into the notebook.


In [ ]:
api_key = os.environ.get("CORE_STACK_API_KEY") or getpass("CoRE Stack API key: ")
api_headers = {"X-API-Key": api_key}


## Read the waterbody API for this tehsil

`get_waterbodies_data_by_admin` returns records keyed by waterbody identifier. Inspect the keys and the available properties.


In [ ]:
response = requests.get(API_URL + "get_waterbodies_data_by_admin/", params=place, headers=api_headers, timeout=180)
response.raise_for_status()
api_waterbodies = response.json()
api_waterbody_table = pd.DataFrame.from_dict(api_waterbodies, orient="index")
api_waterbody_table.head()


## Request the selected waterbody

Pass the same identifier to `get_waterbody_data`. The HTTP status and response show what the service returned.


In [ ]:
response = requests.get(API_URL + "get_waterbody_data/", params={**place, "uid": waterbody_id}, headers=api_headers, timeout=90)
print("HTTP status:", response.status_code)
api_waterbody_response = response.json()
pd.DataFrame.from_dict(api_waterbody_response, orient="index")


## Read its annual and seasonal area fields

Select the same annual and seasonal field names. `reindex` keeps the requested columns visible even when a value is not supplied.


In [ ]:
area_fields = [f"{prefix}_{year}" for year in year_suffixes for prefix in ["area", "k", "kr", "krz"]]
api_area = api_waterbody_table.reindex(index=[waterbody_id], columns=["area_ored"] + area_fields).iloc[0]
api_water_area = pd.DataFrame({
    "Annual area (ha)": [api_area[f"area_{year}"] for year in year_suffixes],
    "Kharif area (ha)": [api_area[f"k_{year}"] * api_area["area_ored"] / 100 for year in year_suffixes],
    "Rabi area (ha)": [api_area[f"kr_{year}"] * api_area["area_ored"] / 100 for year in year_suffixes],
    "Zaid area (ha)": [api_area[f"krz_{year}"] * api_area["area_ored"] / 100 for year in year_suffixes]
}, index=water_area.index)
api_water_area
